# Data Preprocessing and Splitting
---

## Import Libraries

In [3]:
# import libraries
import os
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append('../scripts')
from data_utils import path, column_order, labels
from extract_imgs import find_img_folder, locate_imgs

In [4]:
# load data

data_entry_df = pd.read_pickle('../data/interim/data_entry_df.pkl')
bbox_list = pd.read_pickle('../data/interim/bbox_list.pkl')
patient_data = pd.read_pickle('../data/interim/patient_data.pkl')
train_val_labels = pd.read_csv('../data/labels/train_val_labels.csv', index_col=0)
test_labels = pd.read_csv('../data/labels/test_labels.csv', index_col=0)

## Encode Categorical Variables

In [5]:
# Fit encoders on train_val_labels, then transform both train and test
# Fitting on test data separately would be data leakage

label_encoder_gender = LabelEncoder()
label_encoder_view = LabelEncoder()

train_val_labels["patient_gender"] = label_encoder_gender.fit_transform(train_val_labels["patient_gender"])
print("patient_gender mapping:")
for i, category in enumerate(label_encoder_gender.classes_):
    print(f"  {category}: {i}")

train_val_labels["view_position"] = label_encoder_view.fit_transform(train_val_labels["view_position"])
print("view_position mapping:")
for i, category in enumerate(label_encoder_view.classes_):
    print(f"  {category}: {i}")

train_val_labels[["patient_gender", "view_position"]].head()

patient_gender mapping:
  F: 0
  M: 1
view_position mapping:
  AP: 0
  PA: 1


,patient_gender,view_position
0,1,1
1,1,1
2,1,1
3,1,1
4,1,0


In [6]:
# Apply the fitted encoders to test_labels (transform only — no re-fitting)
test_labels["patient_gender"] = label_encoder_gender.transform(test_labels["patient_gender"])
test_labels["view_position"] = label_encoder_view.transform(test_labels["view_position"])

test_labels[["patient_gender", "view_position"]].head()

,patient_gender,view_position
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1


## Data Splitting
- Note: Data splits have provided by the NIH (train_val_list.txt and test_list.txt), spliting of the train_val set into training and validation sets is still neccessary
- Split "test_labels" and "train_val_labels" into two sets: image data and tabular data

In [7]:
# Replace "image_index" column with file path to each image
test_labels = find_img_folder(test_labels)
train_val_labels = find_img_folder(train_val_labels)

# Add image path to the dataset
def add_image_path(dataset):
    
    paths = []

    for index, img in enumerate(dataset["image_index"]):
        folder = dataset.iloc[index,-1]
        img_path = os.path.join(path, folder, "images", img)
        paths.append(img_path)
    
    dataset.drop(columns="image_index", inplace=True)
    dataset.insert(0, "image_path", paths)
        
# Add to both test and train labels
add_image_path(test_labels)
add_image_path(train_val_labels)

In [8]:
# Stratified 80-20 split by patient_id using iterative_train_test_split
# Ensures both train and val have similar multi-label disease distributions
# Also prevents data leakage (all images from one patient stay in one split)

patient_labels_for_split = train_val_labels.groupby("patient_id")[labels].max().reset_index()
X_patients = patient_labels_for_split["patient_id"].values.reshape(-1, 1)
y_patients = patient_labels_for_split[labels].values

X_train_ids, _, X_val_ids, _ = iterative_train_test_split(X_patients, y_patients, test_size=0.2)

train_ids = X_train_ids.flatten()
val_ids = X_val_ids.flatten()

train_labels = train_val_labels[train_val_labels["patient_id"].isin(train_ids)].reset_index(drop=True)
val_labels = train_val_labels[train_val_labels["patient_id"].isin(val_ids)].reset_index(drop=True)

## Prepare Bounding Box Data

In [9]:
# Rename bounding box column name (to work with find_img_folder)
bbox_list.columns = ['image_index' if col == 'Image Index' else col for col in bbox_list.columns]

# Add image folder 
bbox_list = find_img_folder(bbox_list)

# Add path for each image with bounding box coordinates
add_image_path(bbox_list)
bbox_list.head()

,image_path,Finding Label,x_min,y_min,x_max,y_max,src_folder
0,C:\Users\reala\.cache\kagglehub\datasets\nih-c...,Atelectasis,225.084746,547.019217,86.779661,79.186441,images_006
1,C:\Users\reala\.cache\kagglehub\datasets\nih-c...,Atelectasis,686.101695,131.543498,185.491525,313.491525,images_007
2,C:\Users\reala\.cache\kagglehub\datasets\nih-c...,Atelectasis,221.830508,317.053115,155.118644,216.949153,images_012
3,C:\Users\reala\.cache\kagglehub\datasets\nih-c...,Atelectasis,726.237288,494.951420,141.016949,55.322034,images_007
4,C:\Users\reala\.cache\kagglehub\datasets\nih-c...,Atelectasis,660.067797,569.780787,200.677966,78.101695,images_008


In [ ]:
# Potential issue (bounding box coordinates only conver 8 of the 15 possible target labels)
bbox_list["Finding Label"].value_counts()

Finding Label
Atelectasis     217
Effusion        213
Infiltrate      176
Cardiomegaly    158
Pneumonia       146
Mass            112
Pneumothorax    112
Nodule           82
Name: count, dtype: int64

## Sample Patient Data

In [11]:
# Group images by patient in train and val
train_groups = train_labels.groupby('patient_id')
val_groups = val_labels.groupby('patient_id')

# Group data by patient id
def patient_label_agg(df):
    return df.groupby('patient_id').agg({col: 'max' for col in labels}).reset_index()

train_patient_labels = patient_label_agg(train_labels)
val_patient_labels = patient_label_agg(val_labels)

In [ ]:
# Sample data 

# Define number of sampled patients
num_train_patients = 5000  
num_val_patients = 1250

# Prepare X and y 
X_train_patients = train_patient_labels['patient_id'].values.reshape(-1, 1)
y_train_labels = train_patient_labels[labels].values

# Sample subset of X
X_train_sampled, y_train_sampled, _, _ = iterative_train_test_split(
    X_train_patients, y_train_labels, test_size = 1 - (num_train_patients / len(X_train_patients))
)

train_sampled_patients = X_train_sampled.flatten()

# Sample subset of y
X_val_patients = val_patient_labels['patient_id'].values.reshape(-1, 1)
y_val_labels = val_patient_labels[labels].values

X_val_sampled, y_val_sampled, _, _ = iterative_train_test_split(
    X_val_patients, y_val_labels, test_size = 1 - (num_val_patients / len(X_val_patients))
)

val_sampled_patients = X_val_sampled.flatten()

print(f"Sampled {len(train_sampled_patients)} train patients, {len(val_sampled_patients)} val patients")

Sampled 5040 train patients, 1239 val patients


In [13]:
# Search for sampled patients
train_sampled_df = train_labels[train_labels['patient_id'].isin(train_sampled_patients)].reset_index(drop=True)
val_sampled_df = val_labels[val_labels['patient_id'].isin(val_sampled_patients)].reset_index(drop=True)

# Data integrity check: no patient overlap between train and val
assert len(set(train_sampled_patients) & set(val_sampled_patients)) == 0, "Data leakage: patients appear in both train and val!"

print(f"Training images: {len(train_sampled_df)}")
print(f"Validation images: {len(val_sampled_df)}")
print(f"Test images: {len(test_labels)}")
print(f"\nPer-class training counts:")
print(train_sampled_df[labels].sum().to_string())

Training images: 14858
Validation images: 3734
Test images: 14777

Per-class training counts:
Atelectasis           1293
Cardiomegaly           302
Consolidation          426
Edema                  205
Effusion              1298
Emphysema              214
Fibrosis               211
Hernia                  26
Infiltration          2073
Mass                   670
No Finding            9150
Nodule                 774
Pleural_Thickening     337
Pneumonia              128
Pneumothorax           407


In [14]:
test_cnn = test_labels.drop(columns=["follow_up_number", "patient_id", "patient_age", "patient_gender", "view_position", "src_folder"], axis=1)
test_tab = test_labels.drop(columns=["image_path","patient_id", "src_folder"], axis=1)

train_cnn = train_sampled_df.drop(columns=["follow_up_number", "patient_id", "patient_age", "patient_gender", "view_position", "src_folder"], axis=1)
train_tab = train_sampled_df.drop(columns=["image_path","patient_id", "src_folder"], axis=1)

val_cnn = val_sampled_df.drop(columns=["follow_up_number", "patient_id", "patient_age", "patient_gender", "view_position", "src_folder"], axis=1)
val_tab = val_sampled_df.drop(columns=["image_path","patient_id", "src_folder"], axis=1)

In [15]:
# Scale all 4 tabular features using scaler fitted on train only
target_cols = ["follow_up_number", "patient_age", "patient_gender", "view_position"]

scaler = StandardScaler()
scaler.fit(train_tab[target_cols])

train_tab[target_cols] = scaler.transform(train_tab[target_cols])
val_tab[target_cols] = scaler.transform(val_tab[target_cols])
test_tab[target_cols] = scaler.transform(test_tab[target_cols])

In [16]:
# Split tabular datasets into X and y
test_tab_X = test_tab.drop(columns=[feature for feature in labels], axis=1)
test_tab_y = test_tab[labels]

train_tab_X = train_tab.drop(columns=[feature for feature in labels], axis=1)
train_tab_y = train_tab[labels]

val_tab_X = val_tab.drop(columns=[feature for feature in labels], axis=1)
val_tab_y = val_tab[labels]

In [17]:
# Split CNN dataset into X and y

test_cnn_X = test_cnn.drop(columns=[feature for feature in labels], axis=1)
test_cnn_y = test_cnn[labels]

train_cnn_X = train_cnn.drop(columns=[feature for feature in labels], axis=1)
train_cnn_y = train_cnn[labels]

val_cnn_X = val_cnn.drop(columns=[feature for feature in labels], axis=1)
val_cnn_y = val_cnn[labels]

## Export data

In [18]:
test_cnn_X.to_csv('../data/labels/test_cnn_X.csv')
test_cnn_y.to_csv('../data/labels/test_cnn_y.csv')
test_tab_X.to_csv('../data/labels/test_tab_X.csv')
test_tab_y.to_csv('../data/labels/test_tab_y.csv')

train_cnn_X.to_csv('../data/labels/train_cnn_X.csv')
train_cnn_y.to_csv('../data/labels/train_cnn_y.csv')
train_tab_X.to_csv('../data/labels/train_tab_X.csv')
train_tab_y.to_csv('../data/labels/train_tab_y.csv')

val_cnn_X.to_csv('../data/labels/val_cnn_X.csv')
val_cnn_y.to_csv('../data/labels/val_cnn_y.csv')
val_tab_X.to_csv('../data/labels/val_tab_X.csv')
val_tab_y.to_csv('../data/labels/val_tab_y.csv')